Notebook para generar embeddings
Previamente ya se realizaron para pocos archivos usando un modelos de OpenAI y validando con Qdrant [rag_qdrant](https://github.com/Halsey26/embedding_PerAI/blob/main/rag_qdrant.ipynb)
Sin embargo, ahora son más de 30 archivos pdf, algunos incluso con 300 páginas. Por ende se plantea usar langchain para:
- Chunkenizado
- Embedding
- Almacenamiento - Qdrant
- Función búsqueda
Después se modularizará para detectar los pdfs y obtener los embeddings

Librerias para descargar
- %pip install -qU pypdf
- pip install langchain
- pip install langchain-community
- pip install sentence-transformers
- pip intall tiktoken
- pip install pytesseract pdf2image
- pip install PyPDF2
- sudo apt update
- sudo apt install -y tesseract-ocr


## Función Embedding
Detecta si un pdf ya ha sido procesado. Si en caso no ha sido procesado, se aplica las funciones y se marca como **hecho**.

In [1]:
import os
import hashlib
from langchain_community.document_loaders import PyPDFLoader


In [2]:
def archivo_contenido(archivo):
    if not os.path.exists(archivo): # si no existe el archivo lo crea
        with open(archivo, 'w') as file:
            pass

    # verifica su contenido
    with open(archivo, 'r') as file:
        docs_procesados= list(file.read().splitlines())
    
    # print(f'Documentos procesados: {docs_procesados}')
    return docs_procesados

In [5]:
procesados = archivo_contenido('procesados.txt')

In [5]:
ruta_docs_pdf= '../doc_pdf'
# carpeta_embeddings = ''

def generate_no_procesados(ruta_docs_pdf):
    procesados = archivo_contenido('procesados.txt')
    docs_no_procesados= []
    # verificamos los archivos en carpeta de docs
    for filename in os.listdir(ruta_docs_pdf):
        # verificar si el archivo se encuentra en procesados.txt
        if filename not in procesados:
            # print('El archivo no ha sido procesado')
            ruta_completa= os.path.join(ruta_docs_pdf,filename)
            docs_no_procesados.append(ruta_completa)
        # else:
        #     print('Todos los archivos han sido procesados')

    print(f'Documentos para procesar ({len(docs_no_procesados)}):')
    for doc in docs_no_procesados:
        print(doc)
    return docs_no_procesados


In [4]:
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)

Documentos para procesar (1):
../doc_pdf/prueba.pdf


## Empieza el procesamiento

### Extracción del texto 

In [27]:
from langchain_community.document_loaders import PyPDFLoader

def extraccion_page(ruta):
    loader = PyPDFLoader(ruta)
    pages = loader.load()
    # async for page in loader.alazy_load():
    #     pages.append(page)
    print('✅ Extracción realizada')
    return pages


### Limpieza del texto 

In [41]:
import re

def clean_text(text: str) -> str:
    text = re.sub(r'©.*?\n', '', text)  # remueve símbolos de copyright y similares
    text = re.sub(r'\n+', ' ', text)  # convierte múltiples saltos de línea en espacio
    text = re.sub(r'\s{2,}', ' ', text)  # remueve espacios extra
    text = re.sub(r'\b\d{1,2}:\d{2}\b\s?', '', text) # remueve marca de tiempos
    return text.strip()

### Creacción de la metadata
Estructura planteada:
- documento_id
- nombre documento
- numero pagina
- total_pages

#### **PDF** de imágenes escaneadas

Instalar:
- sudo apt update && sudo apt install -y poppler-utils

In [10]:
from pathlib import Path
ruta_prueba = '../doc_pdf/prueba.pdf'
filename= Path(ruta_prueba).name

In [42]:
from pdf2image import convert_from_path
from PyPDF2 import PdfReader
import pytesseract # para extraer el texto 
from pathlib import Path
import hashlib
import tqdm

def extraccion_ocr_metadata(ruta_completa, filename):
  '''
  Carga, procesa por página y genera cada metadata(streaming)
  '''
  docs_metadata= []
  document_id = hashlib.md5(filename.encode()).hexdigest() # codificamos solo el nombre del archivo

  # carga el pdf
  reader = PdfReader(ruta_completa)
  total_pages = len(reader.pages)

  # convierte cada pag a una imagen
  # imagenes= convert_from_path(ruta_completa)
  # total_pages = len(imagenes)

  print(' Extracción y limpieza ')
  for nro_pag in tqdm.tqdm(range(1,total_pages+1)):
      # carga solo una página como imagen
      imagen= convert_from_path(ruta_completa, first_page= nro_pag, last_page=nro_pag)[0]


      texto = clean_text( pytesseract.image_to_string(imagen)) #definir antes clean text
      
      
      doc= { 
           'text': texto, 
            'metadata' : {
                "document_id": document_id,
                "filename": filename,
                'page_number':nro_pag, 
                'total_pages': total_pages
              }
            }
      docs_metadata.append(doc)

      del imagen #liberar memoria
    
  print('✅ Generación Documentos con Metadata (Limpieza por página)')
  return docs_metadata


In [26]:
extraccion_ocr_metadata(ruta_prueba,filename)

 Extracción y limpieza 


100%|██████████| 3/3 [00:06<00:00,  2.17s/it]

✅ Generación Documentos con Metadata (Limpieza por página)


[{'text': '«E] libro de negocios mas importante e inspirador que he leido nunca.» Tony Schwartz, The New York Times Frederic Laloux llustraciones de Etienne Aopert La guia practica ilustrada del libro que ha revolucionado el management arpa',
  'metadata': {'document_id': '7bbc08499e0ae90368e47ecb9006fce2',
   'filename': 'prueba.pdf',
   'page_number': 1,
   'total_pages': 3}},
 {'text': 'Frederic Laloux trata de combinar los numerosos proyectos que le apasionan con su conviccion intima de que esta destinado a llevar una vida sencilla, en compania de su familia y rodeado de la presencia silenciosa de los arboles. Laloux asesora a lideres corporativos que desean explorar maneras radicalmente nuevas de organizarse. Sus innovadoras investigaciones en el campo de los modelos organizativos emergentes, expuestas en el libro Reinventar las organizaciones, han sido descritas como «revolucionarias», «brillantes», «espectaculares» y «capaces de cambiar el mundo» por algunos de los mas reputados

#### Para pdf normales

In [43]:
from pathlib import Path
import hashlib

def generate_metadata(ruta_completa, pages):
    '''
    Genera metadata de pdf extraccion_page()
    ruta_completa: ruta del documento - str
    pages: 
    '''
    filename = Path(ruta_completa).name
    document_id = hashlib.md5(filename.encode()).hexdigest() # codificamos solo el nombre del archivo
    total_pages = pages[0].metadata['total_pages']
    docs_metadata = []
    print(' Extracción y limpieza ')
    for page in tqdm.tqdm(pages):
        page_number = page.metadata['page_label'] # númeración correcta de la página
        
        metadata = {
            "document_id": document_id,
            "filename": filename,
            "page_number": page_number,
            "total_pages": total_pages,
        }
        page.page_content = clean_text(page.page_content) # cleaned_text = clean_text(page.page_content)
        
        docs_metadata.append(
            {
                'text': page.page_content, #cleaned_text, 
                'metadata': metadata
            }
        )
    print('✅ Generación Documentos con Metadata (Limpieza por página)')
    return docs_metadata

### Generación de embeddings

In [13]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv() # load_dotenv(override=True)  Fuerza que sobreescriba si ya estaba en memoria

api_key=os.getenv('OPENAI_API_KEY')
# api_key
cliente= OpenAI()
cliente

In [15]:
import tiktoken # estimar la cantidad de token
import time

def costo_tokens(tokens):
    costo = tokens*0.02 /10**6 # 1 millon de tokens equivale a 0.02 dólares

    return f'   Tokens: {tokens}\n   Costo Tokens: ${costo:.4f}'


def generate_embedd(docs_metadata):
    print(f'   Generando embedding: ...')
    modelo_openai = "text-embedding-3-small"
    encoding= tiktoken.encoding_for_model(modelo_openai)

    docs_embedd = []
    total_tokens= 0


    for doc in tqdm.tqdm(docs_metadata):
        texto= doc['text']

        # Generación de número de tokens
        tokens= encoding.encode(texto)
        nro_tokens = len(tokens)
        total_tokens += nro_tokens

        doc['metadata']['token']=nro_tokens # añado los tokens a la metadata
    
        start= time.time()
        #  Generación de embeddings
        response = cliente.embeddings.create(
            input= texto, 
            model = modelo_openai
        )
        finish= time.time()
        embedding= response.data[0].embedding
        
        # embedding= modelo_seleccionado.encode(doc['text'], normalize_embeddings= True)
        docs_embedd.append({
            'vector': embedding,  #con openai, directamente el embedding
            'text': texto, 
            'metadata': doc['metadata'] 

        })
    print(costo_tokens(total_tokens))
    print(f'   Tiempo del embedding: {finish-start:.4f} segundos')
    print('✅ Generación de Embeddings')
    return docs_embedd

# comprobar con lo que sale en playground

### Exportación embedding en formato json

In [48]:
ruta_completa

'../doc_pdf/223221647-ECN-BusinessPath-fulldoc.pdf'

In [16]:
# filename = Path(ruta_completa).name

import json
def exportacion_json(docs_embeding,filename):
    with open(f"../json_embedding/{filename}.json", "w", encoding="utf-8") as file:
        json.dump(docs_embeding, file, ensure_ascii=False, indent=2)

    # luego que finalice todo el proceso, hay que realizar una función para agregar el archivo a procesados.txt
    with open('procesados.txt', 'a', encoding='utf-8') as file:
        file.write(filename+"\n")

    print('✅ Exportacción realizada')

# exportacion_json(filename)

## Funcion completa
**def procesamiento():**
- extracion texto
- limpieza por pagina
- creacion de docs_metadata
- obtencion de embedding
- exportación de embedding

In [49]:
import tqdm
prueba = ['../doc_pdf/223221647-ECN-BusinessPath-fulldoc.pdf']

# for ruta_archivo in  tqdm.tqdm(prueba):#docs_no_procesados:
def proceso_completo(docs_no_procesados):
    '''
    Parámetro de entrada: Lista con todas las rutas de los archivos no procesados
    '''
    if docs_no_procesados:
        for ruta_archivo in  docs_no_procesados:#prueba:
            filename = Path(ruta_archivo).name
            print(f'📌 Generando Embeddings para {filename} ...') 
            # definir una funcion para aplicar  
            time1= time.time()
            pags=extraccion_page(ruta_archivo)
            if all(not page.page_content for page in pags):
                print('⚠️ PDF con imágenes detectado. Aplicando OCR ...')
                docs_metadata = extraccion_ocr_metadata(ruta_archivo, filename)
            else:
                docs_metadata = generate_metadata(ruta_archivo, pags)
            
            docs_embedd= generate_embedd(docs_metadata)
            exportacion_json(docs_embedd,filename)
            time3=time.time()
            segundos= time3-time1 
            print(f'\nTiempo total: {segundos:.2f} segundos - {segundos/60:.2f} minutos')
            print('🎉 Realizado: Embeddings Generados Correctamente.\n\n')

    else:
        print('No hay documentos por procesar')
    

In [55]:
proceso_completo(docs_no_procesados)

📌 Generando Embeddings para 223221647-ECN-BusinessPath-fulldoc.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 1938
   Costo Tokens: $0.0000
Tiempo del embedding: 0.5894 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 3.92 segundos
🎉 Realizado: Embeddings Generados Correctamente.



Ya ahora que tengo el embedding demo vamos a modularizar

In [70]:
# actualizamos para docs_no_procesados
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)

Documentos para procesar:
  ['../doc_pdf/601459542-High-Growth-Handbook-PDFDrive-en-Espanol.pdf']


In [ ]:
proceso_completo(docs_no_procesados)

📌 Generando Embeddings para 601459542-High-Growth-Handbook-PDFDrive-en-Espanol.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 190165
   Costo Tokens: $0.0038
   Tiempo del embedding: 0.2444 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 184.87 segundos
🎉 Realizado: Embeddings Generados Correctamente.




In [19]:
proceso_completo(docs_no_procesados)

No hay documentos por procesar


In [20]:
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)

Documentos para procesar:
  ['../doc_pdf/605838498-EMyth-Annual-Plan-2023.pdf']


In [21]:
proceso_completo(docs_no_procesados)

📌 Generando Embeddings para 605838498-EMyth-Annual-Plan-2023.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 4712
   Costo Tokens: $0.0001
   Tiempo del embedding: 0.1757 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 8.74 segundos - 0.15 minutos
🎉 Realizado: Embeddings Generados Correctamente.




In [25]:
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)

Documentos para procesar:
  ['../doc_pdf/383663964-founder-to-ceo-how-to-build-a-great-company-matt-mochary.pdf', '../doc_pdf/465076208-The-Great-CEO-Within.pdf']


In [26]:
proceso_completo(docs_no_procesados)

📌 Generando Embeddings para 383663964-founder-to-ceo-how-to-build-a-great-company-matt-mochary.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 45514
   Costo Tokens: $0.0009
   Tiempo del embedding: 0.1551 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 29.82 segundos - 0.50 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddings para 465076208-The-Great-CEO-Within.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 61821
   Costo Tokens: $0.0012
   Tiempo del embedding: 0.1669 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 64.03 segundos - 1.07 minutos
🎉 Realizado: Embeddings Generados Correctamente.




In [31]:
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)

Documentos para procesar:
  ['../doc_pdf/El juego infinito (Gestión del conocimiento) (Spanish Edition).pdf', '../doc_pdf/Marshall-Ganz-People-Power-and-Change.pdf', '../doc_pdf/354381363-Holocracia.pdf']


In [33]:
proceso_completo(docs_no_procesados)

📌 Generando Embeddings para El juego infinito (Gestión del conocimiento) (Spanish Edition).pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 115394
   Costo Tokens: $0.0023
   Tiempo del embedding: 3.3500 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 77.76 segundos - 1.30 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddings para Marshall-Ganz-People-Power-and-Change.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 12304
   Costo Tokens: $0.0002
   Tiempo del embedding: 0.2302 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 8.02 segundos - 0.13 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddings para 354381363-Holocracia.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Gen

In [39]:
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)

Documentos para procesar:
  ['../doc_pdf/638352159-QUIEN-NO-COMO-Dan-Sullivan.pdf', '../doc_pdf/428339485-Quarterly-Plan-Guide.pdf']


In [40]:
proceso_completo(docs_no_procesados)

📌 Generando Embeddings para 638352159-QUIEN-NO-COMO-Dan-Sullivan.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 75535
   Costo Tokens: $0.0015
   Tiempo del embedding: 0.2153 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 68.96 segundos - 1.15 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddings para 428339485-Quarterly-Plan-Guide.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 2069
   Costo Tokens: $0.0000
   Tiempo del embedding: 0.1946 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 2.62 segundos - 0.04 minutos
🎉 Realizado: Embeddings Generados Correctamente.




In [42]:
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)

Documentos para procesar:
  ['../doc_pdf/The Customer Service Revolution PDF.pdf', '../doc_pdf/593900682-EL-ALMANAKE-DE-NAVAL-RAVIKANT.pdf', '../doc_pdf/544611339-Profit-First-a-Simple-System-to-Transform-Any-Business-From-a-Cash-eating-Monster-to-a-Money-making-Machine-PDFDrive.pdf', '../doc_pdf/The Customer-Funded Business PDF.pdf', '../doc_pdf/767870319-Hyper-Sales-Growth-Jack-Daly.pdf']


In [43]:
proceso_completo(docs_no_procesados)

📌 Generando Embeddings para The Customer Service Revolution PDF.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 19058
   Costo Tokens: $0.0004
   Tiempo del embedding: 0.2501 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 39.61 segundos - 0.66 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddings para 593900682-EL-ALMANAKE-DE-NAVAL-RAVIKANT.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 64615
   Costo Tokens: $0.0013
   Tiempo del embedding: 0.1511 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 42.76 segundos - 0.71 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddings para 544611339-Profit-First-a-Simple-System-to-Transform-Any-Business-From-a-Cash-eating-Monster-to-a-Money-making-Machine-PDFDrive.pdf ...
✅ Extra

In [51]:
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)

Documentos para procesar (5):
../doc_pdf/602446507-The-Great-Game-of-Business-Traslate.pdf
../doc_pdf/639911824-CEO-things-Andreseen.pdf
../doc_pdf/244356764-How-to-Negotiate-Better-Deals-Team-Nanban-pdf.pdf
../doc_pdf/646218768-AULA-01-MARSHALL-GOLDSMITH_portuguese.pdf
../doc_pdf/699641888-Plantillas-Scaling-Up.pdf


In [52]:
proceso_completo(docs_no_procesados)

📌 Generando Embeddings para 602446507-The-Great-Game-of-Business-Traslate.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...


Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 34 0 (offset 0)
Ignoring wrong pointing object 44 0 (offset 0)
Ignoring wrong pointing object 96 0 (offset 0)
Ignoring wrong pointing object 131 0 (offset 0)
Ignoring wrong pointing object 542 0 (offset 0)
Ignoring wrong pointing object 544 0 (offset 0)


   Tokens: 36201
   Costo Tokens: $0.0007
   Tiempo del embedding: 0.1349 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 27.17 segundos - 0.45 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddings para 639911824-CEO-things-Andreseen.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 33028
   Costo Tokens: $0.0007
   Tiempo del embedding: 0.1153 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 35.13 segundos - 0.59 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddings para 244356764-How-to-Negotiate-Better-Deals-Team-Nanban-pdf.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 77031
   Costo Tokens: $0.0015
   Tiempo del embedding: 0.1488 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 77.97 seg

In [56]:
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)

Documentos para procesar (2):
../doc_pdf/476938658-Tribu-de-Mentores-pdf.pdf
../doc_pdf/642218185-BUENO-A-ESTUPENDO-JIM-COLLINS.pdf


In [57]:
proceso_completo(docs_no_procesados)

📌 Generando Embeddings para 476938658-Tribu-de-Mentores-pdf.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 301796
   Costo Tokens: $0.0060
   Tiempo del embedding: 0.1117 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 243.89 segundos - 4.06 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddings para 642218185-BUENO-A-ESTUPENDO-JIM-COLLINS.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 168717
   Costo Tokens: $0.0034
   Tiempo del embedding: 0.2188 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 99.76 segundos - 1.66 minutos
🎉 Realizado: Embeddings Generados Correctamente.




In [58]:
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)

Documentos para procesar (3):
../doc_pdf/The_New_Business_Road_Test.pdf
../doc_pdf/798943251-Sanet-st-Buy-Back-Your-Time-Dan-Martell-1-160-Traducido.pdf
../doc_pdf/599786422-Coleccion-Editorial-Por-Daniel-Marcos.pdf


In [59]:
proceso_completo(docs_no_procesados)

📌 Generando Embeddings para The_New_Business_Road_Test.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 157128
   Costo Tokens: $0.0031
   Tiempo del embedding: 0.1273 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 106.76 segundos - 1.78 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddings para 798943251-Sanet-st-Buy-Back-Your-Time-Dan-Martell-1-160-Traducido.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 65064
   Costo Tokens: $0.0013
   Tiempo del embedding: 0.2658 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 45.03 segundos - 0.75 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddings para 599786422-Coleccion-Editorial-Por-Daniel-Marcos.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpie

In [60]:
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)

Documentos para procesar (1):
../doc_pdf/470174222-Libro-Solo-Una-Cosa.pdf


In [61]:
proceso_completo(docs_no_procesados)

📌 Generando Embeddings para 470174222-Libro-Solo-Una-Cosa.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 88922
   Costo Tokens: $0.0018
   Tiempo del embedding: 0.2151 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 43.50 segundos - 0.72 minutos
🎉 Realizado: Embeddings Generados Correctamente.




In [62]:
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)

Documentos para procesar (6):
../doc_pdf/598958545-LIBRO-TRADUCIDO-Hooked-How-to-Build-Habit-Forming-Products.pdf
../doc_pdf/680189618-who-not-how-en-es.pdf
../doc_pdf/514120050-Impact-X-Workbook-Tool.pdf
../doc_pdf/668523664-Clock-Work-Planeje-Sua-Empresa-Para-Se-Autogerenciar_portuguese.pdf
../doc_pdf/508352943-Simon-Sinek-Lideres-se-Servem-por-Ultimo_portuguese.pdf
../doc_pdf/691700932-Mck-Ceo-Collection-Copy.pdf


In [63]:
proceso_completo(docs_no_procesados)

📌 Generando Embeddings para 598958545-LIBRO-TRADUCIDO-Hooked-How-to-Build-Habit-Forming-Products.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 80266
   Costo Tokens: $0.0016
   Tiempo del embedding: 0.1742 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 45.10 segundos - 0.75 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddings para 680189618-who-not-how-en-es.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 76576
   Costo Tokens: $0.0015
   Tiempo del embedding: 0.6290 segundos
✅ Generación de Embeddings


Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 179 0 (offset 0)


✅ Exportacción realizada

Tiempo total: 54.52 segundos - 0.91 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddings para 514120050-Impact-X-Workbook-Tool.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 851
   Costo Tokens: $0.0000
   Tiempo del embedding: 0.1819 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 4.70 segundos - 0.08 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddings para 668523664-Clock-Work-Planeje-Sua-Empresa-Para-Se-Autogerenciar_portuguese.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 126473
   Costo Tokens: $0.0025
   Tiempo del embedding: 0.1832 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 70.58 segundos - 1.18 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddin

In [66]:
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)

Documentos para procesar (6):
../doc_pdf/EntrepreneursGuideTo10xGrowth.pdf
../doc_pdf/BeginnersGuideToUniqueAbility.pdf
../doc_pdf/8SecretsOfSuccessfulEntrepreneurs.pdf
../doc_pdf/EntrepreneursGuideToProductivity.pdf
../doc_pdf/EntrepreneursGuideToGoalSetting.pdf
../doc_pdf/EntrepreneursGuideToTimeManagement.pdf


In [67]:
proceso_completo(docs_no_procesados)

📌 Generando Embeddings para EntrepreneursGuideTo10xGrowth.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 1936
   Costo Tokens: $0.0000
   Tiempo del embedding: 0.2414 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 2.85 segundos - 0.05 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddings para BeginnersGuideToUniqueAbility.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 2747
   Costo Tokens: $0.0001
   Tiempo del embedding: 0.2427 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 2.42 segundos - 0.04 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddings para 8SecretsOfSuccessfulEntrepreneurs.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens:

In [68]:
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)

Documentos para procesar (2):
../doc_pdf/858783936-El-gran-juego-de-negocios.pdf
../doc_pdf/281722701-Interview-Guide-Topgrading.pdf


In [69]:
proceso_completo(docs_no_procesados)

Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 45 0 (offset 0)
Ignoring wrong pointing object 47 0 (offset 0)
Ignoring wrong pointing object 54 0 (offset 0)
Ignoring wrong pointing object 57 0 (offset 0)
Ignoring wrong pointing object 59 0 (offset 0)
Ignoring wrong pointing object 61 0 (offset 0)
Ignoring wrong pointing object 86 0 (offset 0)
Ignoring wrong pointing object 151 0 (offset 0)
Ignoring wrong pointing object 168 0 (offset 0)
Ignoring wrong pointing object 172 0 (offset 0)
Ignoring wrong pointing object 178 0 (offset 0)


📌 Generando Embeddings para 858783936-El-gran-juego-de-negocios.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 3569
   Costo Tokens: $0.0001
   Tiempo del embedding: 0.1722 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 4.09 segundos - 0.07 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddings para 281722701-Interview-Guide-Topgrading.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 6022
   Costo Tokens: $0.0001
   Tiempo del embedding: 0.2197 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 4.98 segundos - 0.08 minutos
🎉 Realizado: Embeddings Generados Correctamente.




In [35]:
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)
print('')
proceso_completo(docs_no_procesados)

Documentos para procesar (7):
../doc_pdf/ThinkingAboutYourThinking.pdf
../doc_pdf/10xMindExpander.pdf
../doc_pdf/MyPlanForLivingTo156.pdf
../doc_pdf/713272546-CEG-April-Intensive-Dan-Martell-BBYT-Workbook.pdf
../doc_pdf/4cFormula.pdf
../doc_pdf/WantingWhatYouWant.pdf
../doc_pdf/EntrepreneursGuideToASelfManagingCompany.pdf

📌 Generando Embeddings para ThinkingAboutYourThinking.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 14330
   Costo Tokens: $0.0003
   Tiempo del embedding: 0.5936 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 15.92 segundos - 0.27 minutos
🎉 Realizado: Embeddings Generados Correctamente.


📌 Generando Embeddings para 10xMindExpander.pdf ...
✅ Extracción realizada
✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...
   Tokens: 15329
   Costo Tokens: $0.0003
   Tiempo del embedding: 3.6463 segundos
✅ Generación de Embeddings


In [46]:
docs_no_procesados= ['../doc_pdf/prueba.pdf']
print('')
proceso_completo(docs_no_procesados)




📌 Generando Embeddings para prueba.pdf ...
✅ Extracción realizada
⚠️ PDF con imágenes detectado. Aplicando OCR ...
 Extracción y limpieza 


100%|██████████| 3/3 [00:06<00:00,  2.12s/it]


✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...


100%|██████████| 3/3 [00:01<00:00,  1.82it/s]

   Tokens: 431
   Costo Tokens: $0.0000
   Tiempo del embedding: 0.1806 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 8.18 segundos - 0.14 minutos
🎉 Realizado: Embeddings Generados Correctamente.




In [48]:
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)
print('')
proceso_completo(docs_no_procesados)

Documentos para procesar (1):
../doc_pdf/453847022-Reinventar-Las-Organizaciones-Guia-Ilustrada-Laloux-Appert-2017.pdf

📌 Generando Embeddings para 453847022-Reinventar-Las-Organizaciones-Guia-Ilustrada-Laloux-Appert-2017.pdf ...
✅ Extracción realizada
⚠️ PDF con imágenes detectado. Aplicando OCR ...
 Extracción y limpieza 


100%|██████████| 176/176 [12:34<00:00,  4.28s/it]


✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...


100%|██████████| 176/176 [00:51<00:00,  3.44it/s]


   Tokens: 71573
   Costo Tokens: $0.0014
   Tiempo del embedding: 0.1041 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 806.36 segundos - 13.44 minutos
🎉 Realizado: Embeddings Generados Correctamente.




In [52]:
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)
print('')
proceso_completo(docs_no_procesados)

Documentos para procesar (1):
../doc_pdf/739284818-Scaling-Up.pdf

📌 Generando Embeddings para 739284818-Scaling-Up.pdf ...
✅ Extracción realizada
⚠️ PDF con imágenes detectado. Aplicando OCR ...
 Extracción y limpieza 


100%|██████████| 266/266 [33:57<00:00,  7.66s/it]


✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...


100%|██████████| 266/266 [00:58<00:00,  4.54it/s]


   Tokens: 184001
   Costo Tokens: $0.0037
   Tiempo del embedding: 0.1154 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 2097.28 segundos - 34.95 minutos
🎉 Realizado: Embeddings Generados Correctamente.




In [54]:
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)
print('')
proceso_completo(docs_no_procesados)

Documentos para procesar (1):
../doc_pdf/718799418-Simplifica-Tu-Negocio-Miller-Donald-Compress-TOAZ-info.pdf

📌 Generando Embeddings para 718799418-Simplifica-Tu-Negocio-Miller-Donald-Compress-TOAZ-info.pdf ...
✅ Extracción realizada
⚠️ PDF con imágenes detectado. Aplicando OCR ...
 Extracción y limpieza 


100%|██████████| 186/186 [16:52<00:00,  5.44s/it]


✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...


100%|██████████| 186/186 [00:39<00:00,  4.69it/s]


   Tokens: 76787
   Costo Tokens: $0.0015
   Tiempo del embedding: 0.1312 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 1052.90 segundos - 17.55 minutos
🎉 Realizado: Embeddings Generados Correctamente.




In [51]:
docs_no_procesados= generate_no_procesados(ruta_docs_pdf)
print('')
paginas= proceso_completo(docs_no_procesados)

Documentos para procesar (1):
../doc_pdf/Agent AI Early Stage Startup.pdf

📌 Generando Embeddings para Agent AI Early Stage Startup.pdf ...
✅ Extracción realizada
 Extracción y limpieza 


100%|██████████| 327/327 [00:00<00:00, 3756.19it/s]


✅ Generación Documentos con Metadata (Limpieza por página)
   Generando embedding: ...


100%|██████████| 327/327 [01:32<00:00,  3.54it/s]


   Tokens: 159176
   Costo Tokens: $0.0032
   Tiempo del embedding: 0.1751 segundos
✅ Generación de Embeddings
✅ Exportacción realizada

Tiempo total: 138.69 segundos - 2.31 minutos
🎉 Realizado: Embeddings Generados Correctamente.




In [47]:
paginas

[{'text': "uh I think the trade-off if anything for sure would be you know what one goes through in college i I felt I learned equally if not more uh in terms of what I wanted to acquire exposure to a fast growing startup looking at how a venture builder a VC thinks i think having a huge amount of ambition and a good amount of naivity is what helps entrepreneurs get started and having that dreamer within you helps you do things that are bolder one would not conventionally take as well hi I'm Gdoric Chu the co-founder and CEO of Intellect we are a mental health care company serving and building for Asia-Pacific and eventually the world we provide end-to-end mental health support from proactive care all the way towards coaching clinical and distress support we've raised funding from the likes of Tiger Global White Combinator Insignia",
  'metadata': {'document_id': '222c905a619e9e1ee9e12cc6dc343f7e',
   'filename': 'prueba_text.pdf',
   'page_number': '1',
   'total_pages': 2}},
 {'text'

In [65]:
import langchain
import langchain_community